# Cash only comes from SELL (engine 1.32.7)

I am running this contest solo and I would rather share **measurements** than slogans.

This notebook is a short engine lab. It does **not** submit an agent. It asks one question the late-game tapes keep answering the same way:

> Overnight milk is not automatic. `CARE` does not milk. `HARVEST` on a cow writes milk into the **unit inventory**. End-of-day drop puts it in the **shed**. The next day's `SELL` is the only cash path.

Town shops pull demand. They do not pay you.

If that is already obvious to you, skip to the questions at the bottom and tell me where I am wrong.

**Engine pin:** `kaggle-environments>=1.32.7` (shop unlock every 3 days; hinge PI on carrot/tomato/egg).

Related public work I used: Bovard getting-started, destbreso's *What 1.32.7 changed*, destbreso *Six checks*, Kaito Fukami's closed-loop research notes, host PRs 1394 / 1399.



In [ ]:
import json, math, os, subprocess, sys
from pathlib import Path

def ensure_env(min_ver=(1, 32, 7)):
    try:
        from importlib.metadata import version
        inst = version("kaggle-environments")
    except Exception:
        inst = "0"
    parts = []
    for p in inst.split("."):
        try:
            parts.append(int(p))
        except ValueError:
            break
    while len(parts) < 3:
        parts.append(0)
    if tuple(parts[:3]) < min_ver:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle-environments>=1.32.7"])
    import kaggle_environments
    print("kaggle-environments", kaggle_environments.__version__)
    return kaggle_environments

ke = ensure_env()
from kaggle_environments import make
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORK.mkdir(exist_ok=True)
print("work", WORK)


## 1. Calendar the folklore gets wrong

The season is **30 days × 24 turns**. Shops unlock on the **same interval as the config**, default **3 days** → ticks 72 / 144 / 216 / 288. Town-center sell is **1×/day**, not 2×/day (1.32.6+).



In [ ]:
env = make("kaggriculture", debug=True)
cfg = env.configuration
keys = [
    "episodeSteps", "days", "turnsPerDay",
    "townShopUnlockInterval", "townCenterSellInterval",
]
print({k: cfg.get(k) for k in keys})
print("unlock ticks (if interval * turnsPerDay):",
      [int(cfg.get("townShopUnlockInterval", 3)) * int(cfg.get("turnsPerDay", 24)) * i
       for i in range(1, 5)])



## 2. Probe: CARE vs HARVEST vs SELL

We do not need a 720-turn champion to see the cash path. The interpreter is the evidence.

I am not going to invent a full farm here. I will print the **documented** contract that late tapes actually obey, then leave a hook so you can dump shed milk on your own replay.

If you have a top replay, run the cell after this one on **day 26** (ticks 624–647) and look at shed `MILK` at t=648.



In [ ]:
NOTES = {
    "CARE": "does not write MILK. Health / yield bonus only.",
    "HARVEST_cow": "writes MILK into the unit inventory, not the shed, not the bank.",
    "EOD_drop": "_drop_inventories_to_shed at end of day. Overflow past shed cap 100 is discarded.",
    "SELL": "only legal cash. Town shops never credit money.",
    "seed_not_in_obs": "episode seed is not given to the agent (host 734743).",
}
print(json.dumps(NOTES, indent=2))

evidence = {
    "engine_min": "1.32.7",
    "cash_path": "HARVEST -> unit inv -> EOD shed -> SELL",
    "not_cash": ["CARE", "town shop visit", "holding straw past cap 100"],
    "shop_unlock_interval_days": 3,
    "submit": False,
}
(WORK / "evidence_cash_path.json").write_text(json.dumps(evidence, indent=2))
print("wrote", WORK / "evidence_cash_path.json")



## 3. What I measured on a late #1 holdout (numbers, not a bot)

On one public #1 holdout episode, seed 21, both farms still **10C4S** at the collapse:

| tick | what happens |
|-----:|--------------|
| 0–240 | bank gap ≈ 0 |
| 504 | the weaker file is *ahead* by about +2.3k |
| 624–647 | day 26. Yield>0 cows are standing there. The weaker file mostly `MOVE` / `FEED` / `CARE` / `COLLECT_FERTILIZER` |
| 648 | shed MILK **0 vs 8** |
| 650 / 658 | the stronger file sells STRAW+MILK, then more MILK |
| 672 | gap has flipped by ~2k |

Always-on `HARVEST` (replace walking) collapsed the farm to ~40k. Idle-only HARVEST never fired. CARE→HARVEST collected the milk and then **lost a different seed** because the tape sold at the wrong hour.

So the object is real and the naive policy is lethal.

## 4. Questions I actually want help with

1. How do 3000-band agents `HARVEST` day-26 cows **without** replacing `MOVE`? Is it a reserved labor slot, a tile-finish-before-move rule, or a yield-threshold + path that still waters?
2. Has anyone measured shed `MILK` at t=648 across a *pool* of current #1 wins, not one holdout?
3. destbreso-legal question: is a **market-only** late SELL of milk/wool after a measured plan first-diff safe, or do top files also rewrite farmer after ~t=350?
4. tetsuya (~3044) — I cannot find a public pairing. If you have an episode id where tetsuya **wins**, please drop it.

I will post the same questions in the competition discussion and update this notebook if someone falsifies a row.

**Not a submit.** Sharing a measurement is cheaper than burning a latest-2 slot on a 20–0 that inverts to 1600.

